In [28]:

import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
import re
import ast
from tqdm import tqdm
tqdm.pandas(desc="Processing Rows")
import os
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
hf_token = "os.environ['HF_TOKEN']" 
import gc
gc.collect()
import torch
from collections import Counter
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score
torch.cuda.empty_cache()
from scipy.stats import t
import matplotlib.pyplot as plt
import numpy as np

In [2]:
list_cols = [
    "warning type", "time of occurrence",
    "duration of hazard phrase", "frequency of hazard phrase",
    "location", "hazard intensity", "hazard quantity",
    "certainty expression", "protective instruction",
    "impact or consequence"
]

df = pd.read_csv(
    "annotated.csv",
    converters={c: ast.literal_eval for c in list_cols}
) 
df['description']=df['description'].apply(lambda x: re.sub(r"\n+", " ", x).lower())
df.head(3)

,identifier,sent,status,msgType,senderName,event,responseType,urgency,severity,certainty,headline,description,instruction,areaDesc,date,time,description_clean,warning type,time of occurrence,duration of hazard phrase,frequency of hazard phrase,location,hazard intensity,hazard quantity,certainty expression,protective instruction,impact or consequence
0,urn:oid:2.49.0.1.840.0.b65cf952732647052c047cb0efaafe4bf6703291.005.2,2022-12-22T04:28:00.000Z,Actual,Update,NWS Denver CO,Winter Weather Advisory,['Execute'],Expected,Moderate,Likely,Winter Weather Advisory issued December 21 at 9:28PM MST until December 22 at 9:00AM MST by NWS Denver CO,"* what...winds gusting as high as 50 mph, snowfall accumulations up to 3 inches, dangerously cold wind chills as low as 55 below zero. * where...the far northeast plains of colorado, including northeast weld, morgan, washington, logan, phillips and sedgwick counties. * when...for the wind chill warning, until 11 am mst friday. for the winter weather advisory, until 9 am mst thursday. * impacts...plan on slippery roads. blowing snow could significantly reduce visibility. the dangerously cold wind chills could cause frostbite on exposed skin in as little as 5 minutes.","The combination of wind and cold temperatures will create\ndangerously low wind chill values. Avoid outside activities if\npossible. When outside, make sure you wear appropriate clothing, a\nhat, and gloves.\n\nSlow down and use caution while traveling. The latest road\nconditions for Colorado can be obtained by calling 5 1 1 or by going\nto www.cotrip.org.",NaN,2022-12-22,04:28:00,"* what...winds gusting as high as 50 mph, snowfall accumulations up to 3 inches, dangerously cold wind chills as low as 55 below zero. * where...the far northeast plains of colorado, including northeast weld, morgan, washington, logan, phillips and sedgwick counties. * when...for the wind chill warning, until 11 am mst friday. for the winter weather advisory, until 9 am mst thursday. * impacts...plan on slippery roads. blowing snow could significantly reduce visibility. the dangerously cold wind chills could cause frostbite on exposed skin in as little as 5 minutes.",[winter weather advisory],[as little as 5 minutes],"[until 11 am mst friday, until 9 am mst thursday]",[],"[the far northeast plains of colorado, northeast weld, morgan, washington, logan, phillips, sedgwick counties]","[as high as 50 mph, dangerously cold wind chills as low as 55 below zero, dangerously cold wind chills]",[up to 3 inches],"[could, little, can]",[plan on slippery roads],"[blowing snow could significantly reduce visibility, could cause frostbite on exposed skin]"
1,NWS-130404-694658-481250,2014-01-03T17:56:29.000Z,Actual,Update,NWS Buffalo NY,Wind Chill Advisory,['Prepare'],Expected,Moderate,Likely,Wind Chill Advisory issued January 3 at 12:55PM EST expiring January 4 at 4:00AM EST by NWS Buffalo NY,* locations...ontario and livingston counties. * timing...wind chill advisory tonight. * hazards...dangerous low wind chills. * winds...southwest 10 to 15 mph. * wind chill values...as low as 15 below. * impacts...bitterly cold wind chills will pose a risk for frost bite and hypothermia for anyone outside for even short periods of time.,A WIND CHILL ADVISORY IS ISSUED WHEN STRONG WIND WILL COMBINE\nWITH COLD TEMPERATURES TO CREATE DANGEROUSLY COLD CONDITIONS FOR\nEXPOSED SKIN. THE WIND WILL MAKE IT FEEL LIKE IT IS 15 DEGREES\nBELOW ZERO OR COLDER FOR A PERIOD OF SEVERAL HOURS. IF YOU WILL\nBE OUTDOORS USE COMMON SENSE AND DRESS WARMLY...MAKING SURE THAT\nALL EXPOSED SKIN IS COVERED. IF POSSIBLE...AVOID PROLONGED\nEXPOSURE TO THE COLD TO PREVENT FROSTBITE AND HYPOTHERMIA.\n\n,Livingston; Ontario,2014-01-03,17:56:29,* locations...ontario and livingston counties. * timing...wind chill advisory tonight. * hazards...dangerous low wind chills. * winds...southwest 10 to 15 mph. * wind chill values...as low as 15 below. * impacts...bitterly cold wind chills will pose a risk for frost bi

In [3]:
df.columns

Index(['identifier', 'sent', 'status', 'msgType', 'senderName', 'event',
       'responseType', 'urgency', 'severity', 'certainty', 'headline',
       'description', 'instruction', 'areaDesc', 'date', 'time',
       'description_clean', 'warning type', 'time of occurrence',
       'duration of hazard phrase', 'frequency of hazard phrase', 'location',
       'hazard intensity', 'hazard quantity', 'certainty expression',
       'protective instruction', 'impact or consequence'],
      dtype='object')

In [4]:
groups = {
    "intensity": ["hazard intensity", "hazard quantity"],
    "explicit_action": ["warning type", "protective instruction"],
    "epistemic": ["certainty expression"],
    "urgency": ["time of occurrence", "duration of hazard phrase", "frequency of hazard phrase"]
}

def make_ablation_df(df):
    out = []
    for i, r in df.iterrows():
        for g, cols in groups.items():
            text = r["description"]
            for c in cols:
                vals = r[c]
                for v in vals if isinstance(vals, list) else []:
                    text = re.sub(re.escape(str(v)), "", text, flags=re.I)

            out.append([r["description"], text, r["severity"], g])

    return pd.DataFrame(out, columns=[
        "description", "ablated_description", "severity", "ablation"
    ])
ablation_df = make_ablation_df(df)
ablation_df

,description,ablated_description,severity,ablation
0,"* what...winds gusting as high as 50 mph, snowfall accumulations up to 3 inches, dangerously cold wind chills as low as 55 below zero. * where...the far northeast plains of colorado, including northeast weld, morgan, washington, logan, phillips and sedgwick counties. * when...for the wind chill warning, until 11 am mst friday. for the winter weather advisory, until 9 am mst thursday. * impacts...plan on slippery roads. blowing snow could significantly reduce visibility. the dangerously cold wind chills could cause frostbite on exposed skin in as little as 5 minutes.","* what...winds gusting , snowfall accumulations , . * where...the far northeast plains of colorado, including northeast weld, morgan, washington, logan, phillips and sedgwick counties. * when...for the wind chill warning, until 11 am mst friday. for the winter weather advisory, until 9 am mst thursday. * impacts...plan on slippery roads. blowing snow could significantly reduce visibility. the could cause frostbite on exposed skin in as little as 5 minutes.",Moderate,intensity
1,"* what...winds gusting as high as 50 mph, snowfall accumulations up to 3 inches, dangerously cold wind chills as low as 55 below zero. * where...the far northeast plains of colorado, including northeast weld, morgan, washington, logan, phillips and sedgwick counties. * when...for the wind chill warning, until 11 am mst friday. for the winter weather advisory, until 9 am mst thursday. * impacts...plan on slippery roads. blowing snow could significantly reduce visibility. the dangerously cold wind chills could cause frostbite on exposed skin in as little as 5 minutes.","* what...winds gusting as high as 50 mph, snowfall accumulations up to 3 inches, dangerously cold wind chills as low as 55 below zero. * where...the far northeast plains of colorado, including northeast weld, morgan, washington, logan, phillips and sedgwick counties. * when...for the wind chill warning, until 11 am mst friday. for the , until 9 am mst thursday. * impacts.... blowing snow could significantly reduce visibility. the dangerously cold wind chills could cause frostbite on exposed skin in as little as 5 minutes.",Moderate,explicit_action
2,"* what...winds gusting as high as 50 mph, snowfall accumulations up to 3 inches, dangerously cold wind chills as low as 55 below zero. * where...the far northeast plains of colorado, including northeast weld, morgan, washington, logan, phillips and sedgwick counties. * when...for the wind chill warning, until 11 am mst friday. for the winter weather advisory, until 9 am mst thursday. * impacts...plan on slippery roads. blowing snow could significantly reduce visibility. the dangerously cold wind chills could cause frostbite on exposed skin in as little as 5 minutes.","* what...winds gusting as high as 50 mph, snowfall accumulations up to 3 inches, dangerously cold wind chills as low as 55 below zero. * where...the far northeast plains of colorado, including northeast weld, morgan, washington, logan, phillips and sedgwick counties. * when...for the wind chill warning, until 11 am mst friday. for the winter weather advisory, until 9 am mst thursday. * impacts...plan on slippery roads. blowing snow signifitly reduce visibility. the dangerously cold wind chills cause frostbite on exposed skin in as as 5 minutes.",Moderate,epistemic
3,"* what...winds gusting as high as 50 mph, snowfall accumulations up to 3 inches, dangerously cold wind chills as low as 55 below zero. * where...the far northeast plains of colorado, including northeast weld, morgan, washington, logan, phillips and sedgwick counties. * when...for the wind chill warning, until 11 am mst friday. for the winter weather advisory, until 9 am mst thursday. * impacts...plan on slippery roads. blowing snow could significantly reduce visibility. the dangerously cold wind chills could cause frostbite on exposed skin in as little as 5 minutes.","* what...winds gusting as high 

In [80]:
definitions = {
    "intensity": "language describing the intensity, magnitude, or amount of the hazard",
    "explicit_action": "a warning type or explicit protective action people should take",
    "epistemic": "language expressing certainty, uncertainty, possibility, or likelihood",
    "urgency": "information about when the hazard will occur, how long it will last, or how frequently it may occur"
}
choices = ["A", "B"]

template = """Which emergency message indicates the greater threat to life or property?

Message A:
{}

Message B:
{}

Answer only A or B."""

def make_ablation_pairs(df, ablation, seed=42):
    out = df[
        (df["ablation"] == ablation) &
        (df["description"] != df["ablated_description"])
    ].copy()

    rng = np.random.default_rng(seed)
    swap = rng.random(len(out)) < .5

    out["A"] = np.where(swap, out["ablated_description"], out["description"])
    out["B"] = np.where(swap, out["description"], out["ablated_description"])
    out["original"] = np.where(swap, "B", "A")

    return out[["A", "B", "original", "severity", "ablation"]]
    
    
def initalSetup(name):
    model_name = f"/common/data/models/{name}"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    model.eval()
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.eval()
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    choice_ids = {
        c: tokenizer(" " + c, add_special_tokens=False,
                     return_tensors="pt").input_ids[0]
        for c in choices
    }
    return name, model, tokenizer, choice_ids
    
def classify(msg_a, msg_b, model, tokenizer, choice_ids):
    messages = [{
        "role": "user",
        "content": template.format(msg_a, msg_b)
    }]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    pids = tokenizer(
        prompt,
        add_special_tokens=False,
        truncation=True,
        return_tensors="pt"
    ).input_ids[0]

    seqs = [torch.cat([pids, choice_ids[c]]) for c in choices]

    batch = torch.nn.utils.rnn.pad_sequence(
        seqs, batch_first=True,
        padding_value=tokenizer.pad_token_id
    ).to(model.device)

    lengths = torch.tensor([len(x) for x in seqs], device=model.device)
    mask = (
        torch.arange(batch.shape[1], device=model.device)[None, :]
        < lengths[:, None]
    ).long()

    with torch.inference_mode():
        logits = model(input_ids=batch, attention_mask=mask).logits

    scores = {}

    for i, c in enumerate(choices):
        ids = choice_ids[c].to(model.device)
        p = len(pids)

        lp = torch.log_softmax(
            logits[i, p-1:p+len(ids)-1], dim=-1
        )

        scores[c] = lp.gather(
            1, ids.unsqueeze(-1)
        ).sum().item()

    return max(scores, key=scores.get)

from scipy.stats import binomtest

def identify_attribute(text, attr, model, tokenizer, choice_ids):
    prompt = f"""Does this emergency message contain {attr} information?

    Definition: {definitions[attr]}
    
    Message:
    {text}
    
    A: Yes
    B: No
    
    Answer only A or B."""

    pids = tokenizer(
        tokenizer.apply_chat_template(
            [{"role":"user","content":prompt}],
            tokenize=False,
            add_generation_prompt=True
        ),
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids[0]

    seqs = [torch.cat([pids, choice_ids[c]]) for c in choices]
    batch = torch.nn.utils.rnn.pad_sequence(
        seqs, batch_first=True,
        padding_value=tokenizer.pad_token_id
    ).to(model.device)

    with torch.inference_mode():
        logits = model(batch).logits

    scores = {}
    for i,c in enumerate(choices):
        ids = choice_ids[c].to(model.device)
        p = len(pids)
        lp = torch.log_softmax(logits[i,p-1:p+len(ids)-1], dim=-1)
        scores[c] = lp.gather(1,ids.unsqueeze(-1)).sum().item()

    return max(scores, key=scores.get)

def evaluate_detection(df, attr, model, tokenizer, choice_ids):
    cols = groups[attr]

    gold = df[cols].apply(
        lambda r: "A" if any(isinstance(x, list) and len(x) > 0 for x in r) else "B",
        axis=1
    )

    pred = [
        identify_attribute(x, attr, model, tokenizer, choice_ids)
        for x in df["description"]
    ]

    return {
        "accuracy": accuracy_score(gold, pred),
        "balanced_accuracy": balanced_accuracy_score(gold, pred),
        "macro_f1": f1_score(gold, pred, average="macro")
    }
    
def evaluate_ablation(sample, model, tokenizer, choice_ids):
    sample = sample.copy()

    sample["prediction"] = [
        classify(a, b, model, tokenizer, choice_ids)
        for a, b in zip(sample["A"], sample["B"])
    ]
    correct = (sample["prediction"] == sample["original"])
    n = len(sample)
    k = correct.sum()
    ci = binomtest(k, n).proportion_ci(confidence_level=0.95,method="wilson")
    return pd.DataFrame({"value": [n,k / n,ci.low,ci.high]}, index=["n", "original_preference", "ci_lower", "ci_upper"])



severity_choices = {
    "A": "Extreme",
    "B": "Severe",
    "C": "Moderate",
    "D": "Minor",
    "E": "Unknown"
}

def severity_margin(text, model, tokenizer):
    prompt = f"""Classify the severity of this emergency message.
        
        Message:
        {text}
        
        A: Extreme
        B: Severe
        C: Moderate
        D: Minor
        
        Answer only A, B, C, or D."""

    pids = tokenizer(
        tokenizer.apply_chat_template(
            [{"role":"user","content":prompt}],
            tokenize=False,
            add_generation_prompt=True
        ),
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids[0]

    scores = {}

    for c in severity_choices:
        ids = tokenizer(" " + c, add_special_tokens=False,
                        return_tensors="pt").input_ids[0]

        seq = torch.cat([pids, ids]).unsqueeze(0).to(model.device)

        with torch.inference_mode():
            logits = model(seq).logits[0]

        p = len(pids)
        lp = torch.log_softmax(logits[p-1:p+len(ids)-1], dim=-1)
        scores[c] = lp.gather(
            1, ids.to(model.device).unsqueeze(-1)
        ).sum().item()

    ranked = sorted(scores, key=scores.get, reverse=True)

    return {
        "prediction": severity_choices[ranked[0]],
        "margin": scores[ranked[0]] - scores[ranked[1]]
    }

neutral = {
    1: "information",
    2: "additional information",
    3: "additional information provided",
    4: "additional information is provided",
    5: "additional information is provided here",
    6: "additional information is provided here now",
    7: "additional information is provided here for reference",
    8: "additional information is provided here for general reference",
    9: "additional information is provided here for general reference purposes",
    10: "additional information is provided here for general reference purposes only"
}

def replacer(text, listphrases):
    for p in listphrases:
        n = len(str(p).split())
        replacement = " ".join([neutral[10]] * (n // 10) + ([neutral[n % 10]] if n % 10 else []))
        pattern = r"\s+".join(re.escape(x) for x in str(p).split())
        text = re.sub(pattern, replacement, text, count=1, flags=re.I)
    return text


In [48]:
size="large"
if size=="small":
    # all_names=["google--gemma-3-4b-it"]
    all_names=["google--gemma-3-4b-it", "meta-llama--Llama-3.2-3B-Instruct", "microsoft--Phi-4-mini-instruct"]
else:
    all_names=["google--gemma-3-12b-it", "meta-llama--Llama-3.1-8B-Instruct", "microsoft--phi-4"]
ablations=['intensity', 'explicit_action', 'epistemic', 'urgency']

full = pd.DataFrame(index=ablations)
for name in all_names:
    print(name)
    name, model, tokenizer, choice_ids = initalSetup(name)
    for attr in ablations:
        results = evaluate_detection(df, attr, model, tokenizer, choice_ids)
        for metric, value in results.items():
            full.loc[attr, f"{name}_{metric}"] = value
    del model, tokenizer, choice_ids
    torch.cuda.empty_cache()

full

google--gemma-3-12b-it


Loading weights: 100%|██████████| 1065/1065 [00:02<00:00, 413.41it/s] 


meta-llama--Llama-3.1-8B-Instruct


Loading weights: 100%|██████████| 291/291 [00:01<00:00, 177.04it/s]


microsoft--phi-4


Loading weights: 100%|██████████| 243/243 [00:02<00:00, 82.01it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


,google--gemma-3-12b-it_accuracy,google--gemma-3-12b-it_balanced_accuracy,google--gemma-3-12b-it_macro_f1,meta-llama--Llama-3.1-8B-Instruct_accuracy,meta-llama--Llama-3.1-8B-Instruct_balanced_accuracy,meta-llama--Llama-3.1-8B-Instruct_macro_f1,microsoft--phi-4_accuracy,microsoft--phi-4_balanced_accuracy,microsoft--phi-4_macro_f1
intensity,0.8800,0.635270,0.673891,0.8025,0.666907,0.648009,0.8325,0.684502,0.678159
explicit_action,0.5175,0.503280,0.497916,0.5200,0.513232,0.512987,0.5300,0.550645,0.516846
epistemic,0.8625,0.532172,0.526179,0.8675,0.564345,0.579690,0.9475,0.903560,0.894883
urgency,0.9700,0.760000,0.834231,0.8775,0.785333,0.670649,0.8775,0.878667,0.701906


In [46]:
balanced = full[[i for i in full.columns if "balanced" in i or "macro" in i]]
balanced

,google--gemma-3-4b-it_balanced_accuracy,google--gemma-3-4b-it_macro_f1,meta-llama--Llama-3.2-3B-Instruct_balanced_accuracy,meta-llama--Llama-3.2-3B-Instruct_macro_f1,microsoft--Phi-4-mini-instruct_balanced_accuracy,microsoft--Phi-4-mini-instruct_macro_f1
intensity,0.647497,0.674522,0.639719,0.564780,0.517098,0.208896
explicit_action,0.547655,0.527034,0.470332,0.470086,0.503608,0.329082
epistemic,0.516086,0.495495,0.537875,0.217182,0.578717,0.277459
urgency,0.818667,0.776931,0.816000,0.636052,0.573333,0.224747


In [53]:
size="large"
if size=="small":
    # all_names=["google--gemma-3-4b-it"]
    all_names=["google--gemma-3-4b-it", "meta-llama--Llama-3.2-3B-Instruct", "microsoft--Phi-4-mini-instruct"]
else:
    all_names=["google--gemma-3-12b-it", "meta-llama--Llama-3.1-8B-Instruct", "microsoft--phi-4"]
ablations=['intensity', 'explicit_action', 'epistemic', 'urgency']

full = pd.DataFrame(index=ablations)
for name in all_names:
    print(name)
    name, model, tokenizer, choice_ids = initalSetup(name)

    for ablation in ablations:
        pairs_df = make_ablation_pairs(ablation_df, ablation)

        results = evaluate_ablation(pairs_df, model, tokenizer, choice_ids)
        pref = results.loc["original_preference", "value"]
        low  = results.loc["ci_lower", "value"]
        high = results.loc["ci_upper", "value"]
        
        full.loc[ablation, name] = f"{pref:.3f} [{low:.3f}, {high:.3f}]"

    del model, tokenizer, choice_ids
    torch.cuda.empty_cache()

full

google--gemma-3-12b-it


Loading weights: 100%|██████████| 1065/1065 [00:02<00:00, 413.51it/s]


meta-llama--Llama-3.1-8B-Instruct


Loading weights: 100%|██████████| 291/291 [00:01<00:00, 179.31it/s]


microsoft--phi-4


Loading weights: 100%|██████████| 243/243 [00:02<00:00, 81.92it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


,google--gemma-3-12b-it,meta-llama--Llama-3.1-8B-Instruct,microsoft--phi-4
intensity,"0.777 [0.730, 0.818]","0.742 [0.693, 0.786]","0.891 [0.854, 0.920]"
explicit_action,"0.607 [0.541, 0.670]","0.575 [0.509, 0.639]","0.795 [0.736, 0.843]"
epistemic,"0.501 [0.449, 0.554]","0.513 [0.460, 0.566]","0.647 [0.595, 0.696]"
urgency,"0.571 [0.520, 0.620]","0.637 [0.588, 0.684]","0.720 [0.673, 0.763]"


In [54]:
for a in ablation_df["ablation"].unique():
    x = ablation_df[ablation_df["ablation"] == a]
    changed = (x["description"] != x["ablated_description"]).sum()
    print(a, changed, "/", len(x))

intensity 341 / 400
explicit_action 219 / 400
epistemic 343 / 400
urgency 375 / 400


In [58]:
size = "large"

if size == "small":
    all_names = ["google--gemma-3-4b-it", "meta-llama--Llama-3.2-3B-Instruct", "microsoft--Phi-4-mini-instruct"]
else:
    all_names = ["google--gemma-3-12b-it", "meta-llama--Llama-3.1-8B-Instruct", "microsoft--phi-4"]

ablations = ["intensity", "explicit_action", "epistemic", "urgency"]
full = pd.DataFrame(index=ablations)

for name in all_names:
    name, model, tokenizer, choice_ids = initalSetup(name)

    margins = df["description"].apply(lambda x: severity_margin(x, model, tokenizer)["margin"])
    keep = set(df.loc[margins >= margins.median(), "description"])
    temp = ablation_df[ablation_df["description"].isin(keep)]

    for a in ablations:
        r = evaluate_ablation(make_ablation_pairs(temp, a), model, tokenizer, choice_ids)
        full.loc[a, name] = f'{r.loc["original_preference","value"]:.3f} [{r.loc["ci_lower","value"]:.3f}, {r.loc["ci_upper","value"]:.3f}]'

    del model, tokenizer, choice_ids
    torch.cuda.empty_cache()

full

Loading weights: 100%|██████████| 243/243 [00:03<00:00, 72.89it/s]


,google--gemma-3-12b-it,meta-llama--Llama-3.1-8B-Instruct,microsoft--phi-4
intensity,"0.842 [0.779, 0.890]","0.746 [0.679, 0.803]","0.875 [0.816, 0.917]"
explicit_action,"0.711 [0.621, 0.786]","0.569 [0.475, 0.658]","0.758 [0.665, 0.831]"
epistemic,"0.518 [0.448, 0.588]","0.500 [0.430, 0.570]","0.661 [0.591, 0.725]"
urgency,"0.542 [0.469, 0.614]","0.564 [0.495, 0.630]","0.726 [0.657, 0.786]"


In [60]:
df

,identifier,sent,status,msgType,senderName,event,responseType,urgency,severity,certainty,headline,description,instruction,areaDesc,date,time,description_clean,warning type,time of occurrence,duration of hazard phrase,frequency of hazard phrase,location,hazard intensity,hazard quantity,certainty expression,protective instruction,impact or consequence
0,urn:oid:2.49.0.1.840.0.b65cf952732647052c047cb0efaafe4bf6703291.005.2,2022-12-22T04:28:00.000Z,Actual,Update,NWS Denver CO,Winter Weather Advisory,['Execute'],Expected,Moderate,Likely,Winter Weather Advisory issued December 21 at 9:28PM MST until December 22 at 9:00AM MST by NWS Denver CO,"* what...winds gusting as high as 50 mph, snowfall accumulations up to 3 inches, dangerously cold wind chills as low as 55 below zero. * where...the far northeast plains of colorado, including northeast weld, morgan, washington, logan, phillips and sedgwick counties. * when...for the wind chill warning, until 11 am mst friday. for the winter weather advisory, until 9 am mst thursday. * impacts...plan on slippery roads. blowing snow could significantly reduce visibility. the dangerously cold wind chills could cause frostbite on exposed skin in as little as 5 minutes.","The combination of wind and cold temperatures will create\ndangerously low wind chill values. Avoid outside activities if\npossible. When outside, make sure you wear appropriate clothing, a\nhat, and gloves.\n\nSlow down and use caution while traveling. The latest road\nconditions for Colorado can be obtained by calling 5 1 1 or by going\nto www.cotrip.org.",NaN,2022-12-22,04:28:00,"* what...winds gusting as high as 50 mph, snowfall accumulations up to 3 inches, dangerously cold wind chills as low as 55 below zero. * where...the far northeast plains of colorado, including northeast weld, morgan, washington, logan, phillips and sedgwick counties. * when...for the wind chill warning, until 11 am mst friday. for the winter weather advisory, until 9 am mst thursday. * impacts...plan on slippery roads. blowing snow could significantly reduce visibility. the dangerously cold wind chills could cause frostbite on exposed skin in as little as 5 minutes.",[winter weather advisory],[as little as 5 minutes],"[until 11 am mst friday, until 9 am mst thursday]",[],"[the far northeast plains of colorado, northeast weld, morgan, washington, logan, phillips, sedgwick counties]","[as high as 50 mph, dangerously cold wind chills as low as 55 below zero, dangerously cold wind chills]",[up to 3 inches],"[could, little, can]",[plan on slippery roads],"[blowing snow could significantly reduce visibility, could cause frostbite on exposed skin]"
1,NWS-130404-694658-481250,2014-01-03T17:56:29.000Z,Actual,Update,NWS Buffalo NY,Wind Chill Advisory,['Prepare'],Expected,Moderate,Likely,Wind Chill Advisory issued January 3 at 12:55PM EST expiring January 4 at 4:00AM EST by NWS Buffalo NY,* locations...ontario and livingston counties. * timing...wind chill advisory tonight. * hazards...dangerous low wind chills. * winds...southwest 10 to 15 mph. * wind chill values...as low as 15 below. * impacts...bitterly cold wind chills will pose a risk for frost bite and hypothermia for anyone outside for even short periods of time.,A WIND CHILL ADVISORY IS ISSUED WHEN STRONG WIND WILL COMBINE\nWITH COLD TEMPERATURES TO CREATE DANGEROUSLY COLD CONDITIONS FOR\nEXPOSED SKIN. THE WIND WILL MAKE IT FEEL LIKE IT IS 15 DEGREES\nBELOW ZERO OR COLDER FOR A PERIOD OF SEVERAL HOURS. IF YOU WILL\nBE OUTDOORS USE COMMON SENSE AND DRESS WARMLY...MAKING SURE THAT\nALL EXPOSED SKIN IS COVERED. IF POSSIBLE...AVOID PROLONGED\nEXPOSURE TO THE COLD TO PREVENT FROSTBITE AND HYPOTHERMIA.\n\n,Livingston; Ontario,2014-01-03,17:56:29,* locations...ontario and livingston counties. * timing...wind chill advisory tonight. * hazards...dangerous low wind chills. * winds...southwest 10 to 15 mph. * wind chill values...as low as 15 below. * impacts...bitterly cold wind chills will pose a risk for frost bi

In [72]:
stats = []

for group, cols in groups.items():
    vals = pd.concat([df[c].explode() for c in cols]).dropna()
    lengths = vals.astype(str).str.split().str.len()

    per_msg = df.apply(
        lambda r: sum(len(str(p).split()) for c in cols for p in (r[c] if isinstance(r[c], list) else [])),
        axis=1
    )

    stats.append({
        "attribute": group,
        "num_phrases": len(vals),
        "total_words": lengths.sum(),
        "mean_words": lengths.mean(),
        "median_words": lengths.median(),
        "min_words": lengths.min(),
        "max_words": lengths.max(),
        "mean_words_per_msg": per_msg.mean(),
        "median_words_per_msg": per_msg.median(),
        "max_words_per_msg": per_msg.max()
    })

stats = pd.DataFrame(stats).set_index("attribute")
stats

,num_phrases,total_words,mean_words,median_words,min_words,max_words,mean_words_per_msg,median_words_per_msg,max_words_per_msg
attribute,,,,,,,,,
intensity,992,3170,3.195565,3.0,1,11,7.9250,6.0,41
explicit_action,393,1707,4.343511,3.0,1,12,4.2675,2.0,43
epistemic,1418,1424,1.004231,1.0,1,2,3.5600,3.0,22
urgency,951,2711,2.850683,3.0,1,9,6.7775,6.0,31


In [81]:
neutral_df = []

for attr, cols in groups.items():
    temp = df.copy()
    temp["ablated_description"] = temp.apply(lambda r: replacer(r["description"], sum([r[c] for c in cols], [])), axis=1)
    temp["ablation"] = attr
    neutral_df.append(temp[["description", "ablated_description", "severity", "ablation"]])

neutral_df = pd.concat(neutral_df, ignore_index=True)

In [83]:
size="large"
if size=="small":
    # all_names=["google--gemma-3-4b-it"]
    all_names=["google--gemma-3-4b-it", "meta-llama--Llama-3.2-3B-Instruct", "microsoft--Phi-4-mini-instruct"]
else:
    all_names=["google--gemma-3-12b-it", "meta-llama--Llama-3.1-8B-Instruct", "microsoft--phi-4"]
ablations=['intensity', 'explicit_action', 'epistemic', 'urgency']

full = pd.DataFrame(index=ablations)

for name in all_names:
    name, model, tokenizer, choice_ids = initalSetup(name)

    for attr in ablations:
        pairs_df = make_ablation_pairs(neutral_df, attr)
        r = evaluate_ablation(pairs_df, model, tokenizer, choice_ids)
        full.loc[attr, name] = f'{r.loc["original_preference","value"]:.3f} [{r.loc["ci_lower","value"]:.3f}, {r.loc["ci_upper","value"]:.3f}]'

    del model, tokenizer, choice_ids
    torch.cuda.empty_cache()

full

Loading weights: 100%|██████████| 243/243 [00:03<00:00, 72.75it/s]


,google--gemma-3-12b-it,meta-llama--Llama-3.1-8B-Instruct,microsoft--phi-4
intensity,"0.786 [0.739, 0.826]","0.745 [0.696, 0.788]","0.859 [0.818, 0.892]"
explicit_action,"0.653 [0.588, 0.713]","0.603 [0.537, 0.665]","0.858 [0.806, 0.898]"
epistemic,"0.560 [0.507, 0.611]","0.545 [0.492, 0.597]","0.673 [0.622, 0.721]"
urgency,"0.627 [0.577, 0.674]","0.637 [0.588, 0.684]","0.696 [0.648, 0.740]"
